# Kalman filtering: predict, measure, correct

A seeded 2D experiment using the same model as the live Python demo. Run from the repository root. State: `[x, y, vx, vy]`; observations: noisy position. See the browser lab for interactive model mismatch and missing observations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from kalman_model import Settings, Simulation, simulate

settings = Settings(process_noise=1, sensor_noise=1, spikes=False)
data = simulate(duration=8, seed=7, settings=settings)

## The equations

$\hat{x}^- = A\hat{x} + Ga$, $P^- = APA^T + Q$

$K = P^-C^T(CP^-C^T + R)^{\dagger}$

$\hat{x} = \hat{x}^- + K(z-C\hat{x}^-)$

$P = (I-KC)P^-(I-KC)^T + KRK^T$

Here Q is process covariance and R is measurement covariance. The acceleration control is known in this example.

In [ ]:
sim = Simulation()
print("A =", sim.A, "\nG =", sim.G, "\nC =", sim.C)

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
for j, ax in enumerate(axes):
    ax.plot(data["time"], data["measurement"][:, j], ".", alpha=0.2, label="sensor")
    for key in ("truth", "KF", "LB", "IIR", "SMA"):
        ax.plot(data["time"], data[key][:, j], label=key)
    sigma = np.sqrt(np.maximum(0, data["covariance"][:, j, j]))
    ax.fill_between(data["time"], data["KF"][:, j] - 2*sigma, data["KF"][:, j] + 2*sigma, alpha=0.15)
    ax.set_ylabel("xy"[j] + " position")
    ax.grid(alpha=0.3)
axes[0].legend(ncol=6)
axes[-1].set_xlabel("Simulation time [s]")
fig.tight_layout()
plt.show()

## Compare error

Euclidean position RMSE over the full run, including initialization. Repeat with higher sensor noise or scheduled spikes. A matched Kalman model has useful assumptions; it is not automatically robust to unknown outliers.

In [ ]:
for key in ("measurement", "KF", "LB", "IIR", "SMA"):
    rmse = np.sqrt(np.mean(np.sum((data[key] - data["truth"])**2, axis=1)))
    print(f"{key:12s} RMSE: {rmse:.3f}")

## Questions for the audience

1. Why do the simple smoothers lag?
2. Why does the filter estimate velocity when the sensor measures only position?
3. What happens when the motion model or sensor covariance is wrong?
4. Why is a two-sigma 2D ellipse about 86.5% probability rather than 95%?

Use the standalone browser lab to explore questions 3 and 4 visually.